## Install the libraries

In [1]:
!pip install gensim numpy scipy

##  Import Word2Vec

In [2]:
from gensim.models import Word2Vec
import numpy as np


## Create a sample training corpus

In [4]:
from gensim.models import Word2Vec

In [5]:

sentences = [
    ["doctor", "hospital", "patient", "treatment"],
    ["nurse", "patient", "care", "hospital"],
    ["medicine", "doctor", "treatment", "health"],
    ["teacher", "student", "school", "education"],
    ["student", "learning", "knowledge", "teacher"],
    ["school", "classroom", "education", "learning"],
    ["farmer", "crop", "agriculture", "harvest"],
    ["farm", "soil", "crop", "farmer"],
    ["harvest", "agriculture", "food", "production"],
    ["doctor", "medicine", "patient", "health"],
    ["teacher", "knowledge", "student", "learning"],
    ["farmer", "harvest", "food", "crop"]
]

print(f"Loaded {len(sentences)} sample sentences for the corpus.")


Loaded 12 sample sentences for the corpus.


## Train the Word2Vec model

In [6]:
model = Word2Vec(
    sentences=sentences,
    vector_size=10,
    window=2,
    min_count=1,
    sg=1,
    workers=1,
    seed=42,
    epochs=300
)


## Display the embedding for a word

In [7]:
vector = model.wv["doctor"]

print("Embedding for 'doctor':")
print(vector)
print("Vector dimensions:", len(vector))


Embedding for 'doctor':
[ 0.08607478 -0.05652187 -0.00095161 -0.07407539  0.07605276 -0.00306757
 -0.06278493 -0.07574247  0.01085033  0.03536036]
Vector dimensions: 10


## Compare words using cosine similarity

In [8]:
similarity = model.wv.similarity("doctor", "patient")
print("Similarity:", similarity)


Similarity: 0.0067979605


## Find similar words

In [9]:
similar_words = model.wv.most_similar("doctor", topn=5)

for word, score in similar_words:
    print(word, round(score, 3))


medicine 0.691
treatment 0.679
teacher 0.678
farm 0.672
knowledge 0.557


## Sentence Embeddings

##Install the library

In [10]:
!pip install sentence-transformers

##  Load a pretrained model



In [11]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Create customer queries

In [12]:
queries = [
    "My internet connection is very slow.",
    "Why is my WiFi disconnecting frequently?",
    "I cannot connect to the internet.",
    "The mobile application keeps crashing.",
    "I am unable to log in to my account.",
    "My password is not working.",
    "The website takes too long to load.",
    "My internet speed has become very poor."
]

print(f"Loaded {len(queries)} queries.")


Loaded 8 queries.


##Generate sentence embeddings

In [13]:
embeddings = model.encode(queries)
print("Embedding shape:", embeddings.shape)


Embedding shape: (8, 384)


##  Inspect one sentence embedding



In [14]:
print("Original sentence:")
print(queries[0])
print("\nFirst 10 embedding values:")
print(embeddings[0][:15])


Original sentence:
My internet connection is very slow.

First 10 embedding values:
[ 0.05332968  0.00899604  0.02045885 -0.0705438  -0.03673846 -0.07173127
 -0.01319608  0.01111065  0.05202426 -0.00215022  0.02903306  0.02116937
 -0.09937858  0.01067431 -0.00710518]


## Calculate semantic similarity



In [15]:
from sentence_transformers import util

similarity_matrix = util.cos_sim(embeddings, embeddings)
print(similarity_matrix)


tensor([[1.0000, 0.4526, 0.5738, 0.2445, 0.2873, 0.3004, 0.3892, 0.8053],
        [0.4526, 1.0000, 0.3242, 0.2354, 0.1418, 0.1469, 0.1929, 0.3610],
        [0.5738, 0.3242, 1.0000, 0.2821, 0.5297, 0.4031, 0.2931, 0.4687],
        [0.2445, 0.2354, 0.2821, 1.0000, 0.2894, 0.2504, 0.2611, 0.2822],
        [0.2873, 0.1418, 0.5297, 0.2894, 1.0000, 0.6973, 0.2499, 0.2284],
        [0.3004, 0.1469, 0.4031, 0.2504, 0.6973, 1.0000, 0.2152, 0.2120],
        [0.3892, 0.1929, 0.2931, 0.2611, 0.2499, 0.2152, 1.0000, 0.3368],
        [0.8053, 0.3610, 0.4687, 0.2822, 0.2284, 0.2120, 0.3368, 1.0000]])


## Find the most similar customer query

In [16]:
import torch

query_index = 0
scores = similarity_matrix[query_index].clone()
scores[query_index] = -float("inf")
best_match = torch.argmax(scores).item()

print("Customer query:")
print(queries[query_index])
print("\nMost similar query:")
print(queries[best_match])
print("\nSimilarity score:")
print(scores[best_match].item())


Customer query:
My internet connection is very slow.

Most similar query:
My internet speed has become very poor.

Similarity score:
0.8053357601165771


## Contextual Embeddings Using BERT

##  Install libraries



In [17]:
!pip install transformers torch

## Import BERT



In [18]:
import torch
from transformers import AutoTokenizer, AutoModel

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name)
bert_model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

## Define the sentences

In [19]:
sentence1 = "The bat flew out of the cave at night."

sentence2 = "The player used a bat to hit the ball."


## Tokenize the sentences

In [20]:
inputs1 = tokenizer(sentence1, return_tensors="pt")
inputs2 = tokenizer(sentence2, return_tensors="pt")

print(tokenizer.convert_ids_to_tokens(inputs1["input_ids"][0]))
print(tokenizer.convert_ids_to_tokens(inputs2["input_ids"][0]))


['[CLS]', 'the', 'bat', 'flew', 'out', 'of', 'the', 'cave', 'at', 'night', '.', '[SEP]']
['[CLS]', 'the', 'player', 'used', 'a', 'bat', 'to', 'hit', 'the', 'ball', '.', '[SEP]']


## Generate contextual embeddings

In [21]:
with torch.no_grad():
    output1 = bert_model(**inputs1)
    output2 = bert_model(**inputs2)

embedding1 = output1.last_hidden_state
embedding2 = output2.last_hidden_state

print("Sentence 1 shape:", embedding1.shape)
print("Sentence 2 shape:", embedding2.shape)


Sentence 1 shape: torch.Size([1, 12, 768])
Sentence 2 shape: torch.Size([1, 12, 768])


## Extract the representation of "bat"

In [22]:

tokens1 = tokenizer.convert_ids_to_tokens(inputs1["input_ids"][0])
tokens2 = tokenizer.convert_ids_to_tokens(inputs2["input_ids"][0])

bat_index1 = tokens1.index("bat")
bat_index2 = tokens2.index("bat")

bat_vector1 = embedding1[0, bat_index1, :]
bat_vector2 = embedding2[0, bat_index2, :]

print("Bat vector 1 shape:", bat_vector1.shape)
print("Bat vector 2 shape:", bat_vector2.shape)


Bat vector 1 shape: torch.Size([768])
Bat vector 2 shape: torch.Size([768])


## Compare the contextual embeddings

In [23]:
import torch.nn.functional as F

similarity = F.cosine_similarity(
    bat_vector1.unsqueeze(0),
    bat_vector2.unsqueeze(0)
)
print("Cosine similarity:", similarity.item())


Cosine similarity: 0.5624545812606812


## Compare the vectors directly

In [24]:
difference = torch.norm(bat_vector1 - bat_vector2)
print("Euclidean distance:", difference.item())
print("Vectors identical:", torch.allclose(bat_vector1, bat_vector2))


Euclidean distance: 14.3869047164917
Vectors identical: False
